In [1]:
import pandas as pd
import os

# === CONFIGURATION ===
INPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Repo_Summary.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# === LOAD DATA ===
df = pd.read_csv(INPUT_CSV)

# === CLEAN CI PLATFORM FOR GROUPING ===
df['ci_platform_clean'] = df['ci_platform'].fillna("Unknown").str.strip().str.lower()

# === GROUP AND AGGREGATE BY REPO ===
repo_summary = df.groupby('full_name').agg({
    'ci_platform_clean': lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown',
    'device_setup': lambda x: 'Yes' if any(v != 'None' and v != 'Error' for v in x) else 'No',
    'has_trigger': lambda x: 'Yes' if any(v == 'Yes' for v in x) else 'No',
    'Test_Definition': lambda x: 'Yes' if any(v == 'Yes' for v in x) else 'No',
    'parsed_ok': lambda x: all(x),
    'instru_test_status': lambda x: next((v for v in x if v in ['Defective', 'Complete', 'Manual']), 'None'),
}).reset_index()

# === RENAME COLUMNS ===
repo_summary = repo_summary.rename(columns={
    'ci_platform_clean': 'ci_platform',
    'device_setup': 'repo_device_setup',
    'has_trigger': 'repo_has_trigger',
    'Test_Definition': 'repo_test_definition',
    'parsed_ok': 'all_files_parsed_ok',
    'instru_test_status': 'repo_instru_test_status'
})

# === SAVE OUTPUT ===
repo_summary.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Aggregated repository summary saved to:\n{OUTPUT_CSV}")


✅ Aggregated repository summary saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Repo_Summary.csv
